In [1]:
import pandas as pd

# prevent false warning
# https://stackoverflow.com/questions/20625582/how-to-deal-with-settingwithcopywarning-in-pandas
pd.options.mode.chained_assignment = None  # default='warn'

import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import openpyxl # needed for pd to read excel

import math
import re
import datetime
import glob

In [ ]:
%pip install openpyxl

In [190]:
df = pd.read_excel ('1-7Apr2018.xlsx', header=None, usecols=[0, 1, 2, 3, 4], 
                    names=['hour', 'date', 'total', 'day', 'overnight'],
                   skiprows=3)

In [191]:
# extract date
date_range = df.at[0, 'date']

def get_start_date(text):
    m = re.search("From: (\d{4}-\d{2}-\d{2})", text)
    
    if not m:
        print("No Start date found")
        return None
    
    return datetime.date(*map(int, m[1].split('-')))

def get_end_date(text):
    m = re.search("To: (\d{4}-\d{2}-\d{2})", text)
    
    if not m:
        print("No Start date found")
        return None
    
    return datetime.date(*map(int, m[1].split('-')))

    
start_date = get_start_date(date_range)
end_date   = get_end_date(date_range)

# drop empty start rows
df = df.iloc[6:]
df = df.reset_index(drop=True)

In [192]:
df['start_date'] = start_date
df['end_date']   = end_date

In [193]:
rows_to_remove = [] # inidizes of rows containing cmap names
current_camp = None

df['camp'] = None
for i, row in df.iterrows():

    # check if totals are correct
    if row['hour'] == 'TOTAL':
        dfx = df[df['camp'] == current_camp]
  
        total = dfx['total'].sum()
        if total != row['total']:
            print('Sum of total for camp {} not correct, {} given, expected {}'.format(current_camp, row['total'], total))
        
        day = dfx['day'].sum()
        if day != row['day']:
            print('Sum of total for camp {} not correct, {} given, expected {}'.format(current_camp, row['day'], day))
        
        overnight = dfx['overnight'].sum()
        if overnight != row['overnight']:
            print('Sum of total for camp {} not correct, {} given, expected {}'.format(current_camp, row['overnight'], overnight))
        rows_to_remove.append(i)
        continue
    
    # check if new camp name
    if math.isnan(row['total']) and math.isnan(row['day']) and math.isnan(row['overnight']):
        current_camp = row['hour']
        rows_to_remove.append(i)
        continue
        
    
    
    row_sum = row['day'] + row['overnight']
    if (row_sum != row['total']):
        print('Sum of day/overnight does not match total for camp {} at hour {}'.format(current_camp, row['hour']))
      

    
    df.at[i, 'camp'] = current_camp
    
# drop meta rows
df2 = df.drop(df.index[rows_to_remove]).reset_index(drop=True)

In [196]:
# Liandi: For camps situated very close to a gate, add the numbers to the gate arrivals

# CROCODILE BRIDGE REST CAMP – ADD the numbers to the CROCODILE BRIDGE GATE arrivals.
dfx = df2[df2['camp'] == 'CROCODILE BRIDGE REST CAMP']
for i, row in dfx.iterrows():
    
    loc = df2.loc[(df2['camp'] == 'CROCODILE BRIDGE GATE') & (df2['hour'] == row['hour'])]
    
    # add camp to gate for existing hours
    if not loc.empty:        
        df2.at[loc.index[0], 'total']     += row['total']
        df2.at[loc.index[0], 'day']       += row['day']
        df2.at[loc.index[0], 'overnight'] += row['overnight']
    else:
        # hour did not exist for gate, add it by overwrioting camp name with gate name ;)
        df2.at[i, 'camp'] = 'CROCODILE BRIDGE GATE'

# ORPEN REST CAMP – ADD the numbers to the ORPEN GATE arrivals.
dfx = df2[df2['camp'] == 'ORPEN REST CAMP']
for i, row in dfx.iterrows():
    
    loc = df2.loc[(df2['camp'] == 'ORPEN GATE') & (df2['hour'] == row['hour'])]
    
    # add camp to gate for existing hours
    if not loc.empty:        
        df2.at[loc.index[0], 'total']     += row['total']
        df2.at[loc.index[0], 'day']       += row['day']
        df2.at[loc.index[0], 'overnight'] += row['overnight']
    else:
        # hour did not exist for gate, add it by overwrioting camp name with gate name ;)
        df2.at[i, 'camp'] = 'ORPEN GATE'
    

In [212]:
# For all other rest camps, ignore the numbers. They are due to users in the system who, 
# after helping visitors look for accommodation in the rest camp, forgot to change their 
# location back to the gate. Excluding them will not make any material difference.

# -> remove everything that is a CAMP
df3 = df2[~df2.camp.str.contains('CAMP')]
df3 = df3[~df3.camp.str.contains('AIRPORT')] # exclude aiurport aswell


# lodges as well?
df3 = df2[~df2.camp.str.contains('LODGE')]


df3 = df3.reset_index(drop=True)

In [207]:
# normalize hours
def get_start_time(s):
    m = s.split('-')
    return m[0].strip()
def get_end_time(s):
    m = s.split('-')
    return m[1].strip()

df3['start_time'] = df3['hour'].apply(get_start_time)
df3['end_time'] = df3['hour'].apply(get_end_time)

In [208]:
df3


,hour,date,total,day,overnight,start_date,end_date,camp,start_time,end_time
0,06:00 - 06:59,NaN,2519,2284,235,2018-04-01,2018-04-07,CROCODILE BRIDGE GATE,06:00,06:59
1,07:00 - 07:59,NaN,1689,1523,166,2018-04-01,2018-04-07,CROCODILE BRIDGE GATE,07:00,07:59
2,08:00 - 08:59,NaN,1036,951,85,2018-04-01,2018-04-07,CROCODILE BRIDGE GATE,08:00,08:59
3,09:00 - 09:59,NaN,835,697,138,2018-04-01,2018-04-07,CROCODILE BRIDGE GATE,09:00,09:59
4,10:00 - 10:59,NaN,588,478,110,2018-04-01,2018-04-07,CROCODILE BRIDGE GATE,10:00,10:59
...,...,...,...,...,...,...,...,...,...,...
119,13:00 - 13:59,NaN,251,219,32,2018-04-01,2018-04-07,PUNDA MARIA GATE,13:00,13:59
120,14:00 - 14:59,NaN,338,280,58,2018-04-01,2018-04-07,PUNDA MARIA GATE,14:00,14:59
121,15:00 - 15:59,NaN,77,45,32,2018-04-01,2018-04-07,PUNDA MARIA GATE,15:00,15:59
122,16:00 - 16:59,NaN,37,13,24,2018-04-01,2018-04-07,PUNDA MARIA GATE,16:00,16:59


In [211]:
str(df3.at[0, 'start_date'])

'2018-04-01'